In [76]:
import pandas as pd
import requests
import json
import re
from io import StringIO

url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

fixtures = []

for table in tables:
    columns = list(table.columns)

    # We only want tables that look like:
    # Team A | Match X | Team B
    if len(columns) == 3 and "Match" in str(columns[1]):
        home_team = str(columns[0]).strip()
        match_text = str(columns[1]).strip()
        away_team = str(columns[2]).strip()

        match_number = re.search(r"\d+", match_text)

        if not match_number:
            continue

        fixtures.append({
            "matchId": int(match_number.group()) if match_number else None,
            "homeTeam": home_team,
            "awayTeam": away_team,
            "homeTeamKey": home_team.lower().replace(" ", "_"),
            "awayTeamKey": away_team.lower().replace(" ", "_"),
            "matchImportance": 5,
            "rivalryScore": 5
        })

fixtures = sorted(fixtures, key=lambda x: x["matchId"])

with open("fixtures.json", "w") as f:
    json.dump({"fixtures": fixtures}, f, indent=2)

print(f"Saved {len(fixtures)} fixtures to fixtures.json")


Saved 102 fixtures to fixtures.json


In [123]:
# Import libraries
import json
from pathlib import Path

# Load and open the data
BASE_DIR = Path("/Users/jasonrobert/Documents/Data_Analysis/projects/wcExcite/data")

with open(BASE_DIR / "teams.json", "r") as f:
    teams = json.load(f)

with open(BASE_DIR / "fixtures.json", "r") as f:
    fixtures_data = json.load(f)

fixtures = fixtures_data["fixtures"]

# UTILITY FUNCTIONS AND VARIABLE
# 1. Normalize values
def normalize(value, min_value, max_value):
    score = ((value - min_value) / (max_value - min_value)) * 10
    return max(0, min(score, 10))

# 2. Check if teams has World Cup stats
def has_wc_stats(home_team, away_team):
    required_keys = [
        "wcGoalsForPerMatch",
        "wcGoalsAgainstPerMatch",
        "wcShotsPerMatch",
        "wcShotsOnTargetPerMatch",
        "wcShotsAgainstPerMatch",
        "wcShotsOnTargetAgainstPerMatch",
        "wcXg",
        "wcXgA",
    ]

    return all(
        key in home_team and key in away_team
        for key in required_keys
    )
# Weight for pre-tournament and tournament excitement score
PRE_WEIGHT = 0.35
WC_WEIGHT = 0.65

# 3. Watch tier function
def get_watch_tier(score):
        if score >= 75:
            return "Match of the tournament candidate"
        elif score >= 65:
            return "Must watch"
        elif score >= 60:
            return "Strong watch"
        elif score >=50:
            return "Background watch"
        else:
            return "Skippable unless you support them"

scored_fixtures = []

# SCORING FUNCTIONS

# 4. Calculate excitement score with pre-tournament stats
def calculate_excitement(home_key, away_key, match_importance, rivalry_score):
    home_team = teams[home_key]
    away_team = teams[away_key]

    avg_goals_for = (
        home_team["goalsForPerMatch"] + away_team["goalsForPerMatch"]
    ) / 2

    avg_goals_against = (
        home_team["goalsAgainstPerMatch"] + away_team["goalsAgainstPerMatch"]
    ) / 2

    avg_star_power = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    elo_difference = abs(home_team["eloRating"] - away_team["eloRating"])

    attacking_score = normalize(avg_goals_for, 0, 3)
    chaos_score = normalize(avg_goals_against, 0, 3)
    # Smaller Elo gap = more balanced game = higher score
    competitive_balance = 10 - normalize(elo_difference, 0, 600)

    excitement_score = (
        0.25 * attacking_score +
        0.20 * chaos_score +
        0.20 * avg_star_power +
        0.15 * competitive_balance +
        0.10 * match_importance +
        0.10 * rivalry_score) * 10

    team_quality_score = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    quality_floor = team_quality_score * 7.5

    excitement_score = max(excitement_score, quality_floor)

    return {
        "finalScore": round(excitement_score, 1),
        "breakdown": {
            "attackingScore": round(attacking_score, 2),
            "chaosScore": round(chaos_score, 2),
            "starPowerScore": round(avg_star_power, 2),
            "competitiveBalance": round(competitive_balance, 2),
            "matchImportance": match_importance,
            "rivalryScore": rivalry_score
        }
    }
    
# 5. Calculate excitement score with tournament stats
def calculate_excitement_wc_stats(home_key, away_key, match_importance, rivalry_score):
    home_team = teams[home_key]
    away_team = teams[away_key]

    avg_goals_for = (
        home_team["wcGoalsForPerMatch"] + away_team["wcGoalsForPerMatch"]
    ) / 2
    avg_goals_against = (
        home_team["wcGoalsAgainstPerMatch"] + away_team["wcGoalsAgainstPerMatch"]
    ) / 2

    avg_xg = (home_team["wcXg"] + away_team["wcXg"]) / 2
    avg_xg_against = (home_team["wcXgA"] + away_team["wcXgA"]) / 2

    avg_shots_per_match = (home_team["wcShotsPerMatch"] + away_team["wcShotsPerMatch"]) / 2
    avg_shots_on_target_per_match = (home_team["wcShotsOnTargetPerMatch"] + away_team["wcShotsOnTargetPerMatch"]) / 2

    avg_shots_against_per_match = (home_team["wcShotsAgainstPerMatch"] + away_team["wcShotsAgainstPerMatch"]) / 2
    avg_shots_on_target_against_per_match = (home_team["wcShotsOnTargetAgainstPerMatch"] + away_team["wcShotsOnTargetAgainstPerMatch"]) / 2

    goals_score = normalize(avg_goals_for, 0, 3)
    xg_score = normalize(avg_xg, 0, 3)
    shots_score = normalize(avg_shots_per_match, 5, 20)
    sot_score = normalize(avg_shots_on_target_per_match, 1, 8)

    attacking_score = (
        0.35 * goals_score + 
        0.35 * xg_score + 
        0.15 * shots_score + 
        0.15 * sot_score
    )

    goals_against_score = normalize(avg_goals_against, 0, 3)
    xga_score = normalize(avg_xg_against, 0, 3)
    shots_against_score = normalize(avg_shots_against_per_match, 5, 20)
    sot_against_score = normalize(avg_shots_on_target_against_per_match, 1, 8)

    chaos_score = (
        0.35 * goals_against_score +
        0.35 * xga_score +
        0.15 * shots_against_score +
        0.15 * sot_against_score
    )

    avg_star_power = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    elo_difference = abs(home_team["eloRating"] - away_team["eloRating"])

    competitive_balance = 10 - normalize(elo_difference, 0, 600)

    excitement_score = (
        0.35 * attacking_score +
        0.15 * chaos_score +
        0.15 * avg_star_power +
        0.10 * competitive_balance +
        0.15 * match_importance +
        0.10 * rivalry_score) * 10

    team_quality_score = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    quality_floor = team_quality_score * 7.5

    excitement_score = max(excitement_score, quality_floor)

    return {
        "finalScore": round(excitement_score, 1),
        "breakdown": {
            "attackingScore": round(attacking_score, 2),
            "chaosScore": round(chaos_score, 2),
            "starPowerScore": round(avg_star_power, 2),
            "competitiveBalance": round(competitive_balance, 2),
            "matchImportance": match_importance,
            "rivalryScore": rivalry_score
        }
    }

# MAIN SCORING LOOP
for fixture in fixtures:
    home_key = fixture["homeTeamKey"]
    away_key = fixture["awayTeamKey"]

    teams_are_known = home_key in teams and away_key in teams

    if not teams_are_known:
        scored_fixture = {
            **fixture,
            "homeTeamName": fixture.get("homeTeam", home_key),
            "awayTeamName": fixture.get("awayTeam", away_key),
            "excitementScore": None,
            "watchTier": "TBD",
            "scoreBreakdown": None,
            "starPlayers": [],
            "whyWatch": fixture.get("whyWatch", ""),
            "status": fixture.get("status", "tbd")
        }

        scored_fixtures.append(scored_fixture)
        continue

    home_team = teams[home_key]
    away_team = teams[away_key]

    pre_result = calculate_excitement(
        home_key,
        away_key,
        fixture.get("matchImportance", 5),
        fixture.get("rivalryScore", 5)
    )

    is_group_stage = fixture.get("stage", "Group Stage") == "Group Stage"

    if is_group_stage:
        final_score = pre_result["finalScore"]
        score_breakdown = pre_result["breakdown"]
        wc_result = None
    elif has_wc_stats(home_team, away_team):
        wc_result = calculate_excitement_wc_stats(
            home_key,
            away_key,
            fixture.get("matchImportance", 5),
            fixture.get("rivalryScore", 5)
        )

        final_score = (
            PRE_WEIGHT * pre_result["finalScore"] +
            WC_WEIGHT * wc_result["finalScore"]
        )
        score_breakdown = wc_result["breakdown"]
    else:
        wc_result = None
        final_score = pre_result["finalScore"]
        score_breakdown = pre_result["breakdown"]

    scored_fixture = {
        **fixture,
        "homeTeamName": home_team["teamName"],
        "awayTeamName": away_team["teamName"],
        "homeElo": home_team["eloRating"],
        "awayElo": away_team["eloRating"],
        "homeStarPower": home_team["starPowerScore"],
        "awayStarPower": away_team["starPowerScore"],
        "starPlayers": (
            home_team.get("starPlayers", []) +
            away_team.get("starPlayers", [])
        ),
        "whyWatch": fixture.get("whyWatch", ""),
        "storylineScore": fixture.get("storylineScore", None),
        "storylines": fixture.get("storylines", []),
        "excitementScore": round(final_score, 1),
        "preTournamentScore": pre_result["finalScore"],
        "tournamentFormScore": wc_result["finalScore"] if wc_result else None,
        "watchTier": get_watch_tier(final_score),
        "scoreBreakdown": score_breakdown,
    }

    scored_fixtures.append(scored_fixture)


scored_fixtures = sorted(
    scored_fixtures,
    key=lambda x:x["excitementScore"] if x["excitementScore"] is not None else -1,
    reverse=True
)

with open(BASE_DIR / "scored_fixtures.json", "w") as f:
    json.dump({"fixtures": scored_fixtures}, f, indent=2)

for fixture in scored_fixtures:
    print(
        f"{fixture['homeTeamName']} vs {fixture['awayTeamName']}: "
        f"{fixture['excitementScore']} - {fixture['watchTier']}"
    )

Norway vs France: 68.6 - Must watch
England vs Croatia: 67.9 - Must watch
Norway vs Senegal: 67.8 - Must watch
Brazil vs Norway: 66.7 - Must watch
Belgium vs Senegal: 66.6 - Must watch
Netherlands vs Japan: 65.6 - Must watch
Colombia vs Portugal: 64.7 - Strong watch
Brazil vs Japan: 64.2 - Strong watch
Portugal vs Croatia: 63.8 - Strong watch
France vs Senegal: 63.6 - Strong watch
Netherlands vs Morocco: 63.3 - Strong watch
Belgium vs Iran: 62.6 - Strong watch
United States vs Australia: 61.7 - Strong watch
Netherlands vs Sweden: 61.7 - Strong watch
France vs Sweden: 60.8 - Strong watch
Belgium vs Egypt: 60.6 - Strong watch
Ivory Coast vs Norway: 60.4 - Strong watch
Australia vs Turkey: 60.1 - Strong watch
Ecuador vs Germany: 60.0 - Strong watch
Uruguay vs Spain: 60.0 - Strong watch
Algeria vs Austria: 59.9 - Background watch
Germany vs Ivory Coast: 59.7 - Background watch
Japan vs Sweden: 59.4 - Background watch
South Korea vs Czechia: 58.9 - Background watch
New Zealand vs Egypt: 58.

In [ ]:
import json
from pathlib import Path

BASE_DIR = Path("/Users/jasonrobert/Documents/Data_Analysis/projects/wcExcite/data")

with open(BASE_DIR / "teams.json", "r") as f:
    teams = json.load(f)

def normalize(value, min_value, max_value):
    score = ((value - min_value) / (max_value - min_value)) * 10
    return max(0, min(score, 10))
    
# Tournament stats formula

def calculate_excitement_wc_stats(home_key, away_key, match_importance, rivalry_score):
    home_team = teams[home_key]
    away_team = teams[away_key]

    avg_goals_for = (
        home_team["wcGoalsForPerMatch"] + away_team["wcGoalsForPerMatch"]
    ) / 2
    avg_goals_against = (
        home_team["wcGoalsAgainstPerMatch"] + away_team["wcGoalsAgainstPerMatch"]
    ) / 2

    avg_xg = (home_team["wcXg"] + away_team["wcXg"]) / 2
    avg_xg_against = (home_team["wcXgA"] + away_team["wcXgA"]) / 2

    avg_shots_per_match = (home_team["wcShotsPerMatch"] + away_team["wcShotsPerMatch"]) / 2
    avg_shots_on_target_per_match = (home_team["wcShotsOnTargetPerMatch"] + away_team["wcShotsOnTargetPerMatch"]) / 2

    avg_shots_against_per_match = (home_team["wcShotsAgainstPerMatch"] + away_team["wcShotsAgainstPerMatch"]) / 2
    avg_shots_on_target_against_per_match = (home_team["wcShotsOnTargetAgainstPerMatch"] + away_team["wcShotsOnTargetAgainstPerMatch"]) / 2

    goals_score = normalize(avg_goals_for, 0, 3)
    xg_score = normalize(avg_xg, 0, 3)
    shots_score = normalize(avg_shots_per_match, 5, 20)
    sot_score = normalize(avg_shots_on_target_per_match, 1, 8)

    attacking_score = (
        0.35 * goals_score + 
        0.35 * xg_score + 
        0.15 * shots_score + 
        0.15 * sot_score
    )

    goals_against_score = normalize(avg_goals_against, 0, 3)
    xga_score = normalize(avg_xg_against, 0, 3)
    shots_against_score = normalize(avg_shots_against_per_match, 5, 20)
    sot_against_score = normalize(avg_shots_on_target_against_per_match, 1, 8)

    chaos_score = (
        0.35 * goals_against_score +
        0.35 * xga_score +
        0.15 * shots_against_score +
        0.15 * sot_against_score
    )

    avg_star_power = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    elo_difference = abs(home_team["eloRating"] - away_team["eloRating"])

    competitive_balance = 10 - normalize(elo_difference, 0, 600)

    excitement_score = (
        0.35 * attacking_score +
        0.15 * chaos_score +
        0.15 * avg_star_power +
        0.10 * competitive_balance +
        0.15 * match_importance +
        0.10 * rivalry_score) * 10

    team_quality_score = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    quality_floor = team_quality_score * 7.5

    excitement_score = max(excitement_score, quality_floor)

    return {
        "finalScore": round(excitement_score, 1),
        "breakdown": {
            "attackingScore": round(attacking_score, 2),
            "chaosScore": round(chaos_score, 2),
            "starPowerScore": round(avg_star_power, 2),
            "competitiveBalance": round(competitive_balance, 2),
            "matchImportance": match_importance,
            "rivalryScore": rivalry_score
        }
    }

result = calculate_excitement_wc_stats(
    "canada",
    "south africa",
    match_importance=6,
    rivalry_score=5
    )

print(result)

Goals: 5.533333333333333
xG: 5.616666666666666
Shots: 6.776666666666666
SOT: 5.928571428571429
Attacking: 5.808285714285714
Chaos: 3.051761904761905
Competitive: 5.5
{'finalScore': 51.9, 'breakdown': {'attackingScore': 5.81, 'chaosScore': 3.05, 'starPowerScore': 5.0, 'competitiveBalance': 5.5, 'matchImportance': 6, 'rivalryScore': 5}}
